# Compton and double Compton scattering from the Feynman diagrams

J. Chluba, *Spectral distortions of the cosmic microwave background* (PhD thesis, 2005) uses Compton
scattering and double Compton emission. Here both are computed from their Feynman diagrams with feynsage,
with FORM for the Dirac traces, and compared with the formulas the thesis uses:

1. Compton scattering: the sum over spins and polarisations of $|\mathcal M|^2$, Klein–Nishina and Thomson
2. Double Compton scattering, six diagrams: Mandl and Skyrme's $X$ (thesis eq. D.1)
3. The soft limit of double Compton: the $1/\omega_2$ infrared divergence

The same calculations done with FeynCalc are in `compton_dc_feyncalc.wl`. Run this notebook with the
SageMath kernel, from this folder or from the top folder of the repository.

In [1]:
import sys, os
# find the repository (this notebook may be opened from examples/chluba or from the top folder)
root = os.getcwd()
while not os.path.isfile(os.path.join(root, "feynsage", "__init__.py")) and root != "/":
    root = os.path.dirname(root)
sys.path.insert(0, root)
sys.path.insert(0, os.path.join(root, "examples", "chluba"))
from feynsage import *
from double_compton import dc_table, dc_msq, mandl_skyrme_X, eikonal

## 1. Compton scattering

$e(p) + \gamma(k) \to e(p') + \gamma(k')$ has two diagrams: the electron absorbs $k$ first ($s$-channel) or
emits $k'$ first ($u$-channel). Summed over spins and polarisations,
$$\sum|\mathcal M|^2 = e^4\Big[\frac{T_{ss}}{(s - m^2)^2} + \frac{T_{uu}}{(u - m^2)^2} + \frac{2T_{su}}{(s - m^2)(u - m^2)}\Big]$$
with three traces $T$. Each photon polarisation sum gives $-g_{\mu\nu}$ and the two minus signs cancel.
We write $p' = p + k - k'$ inside the traces, so only $p\cdot k$ and $p\cdot k'$ are needed:
$s - m^2 = 2p\cdot k$, $u - m^2 = -2p\cdot k'$ and $k\cdot k' = p\cdot k - p\cdot k'$ (because $p'^2 = m^2$).
Each trace is one `form.compute` call (see `examples/feynsage_first_steps.ipynb` for its arguments).

In [2]:
m, pk, pkp = var('m pk pkp')                     # pk = p.k and pkp = p.k'
rules = {"p.p": "m^2", "k.k": 0, "kp.kp": 0, "p.k": "pk", "p.kp": "pkp", "k.kp": "pk - pkp"}
V = ["p", "k", "kp"]
pp = "p + k - kp + m"                            # slashed p' plus m
T_ss = form.compute(vectors=V, lines=[[pp, "nu", "p + k + m", "mu", "p + m", "mu", "p + k + m", "nu"]], rules=rules)
T_uu = form.compute(vectors=V, lines=[[pp, "mu", "p - kp + m", "nu", "p + m", "nu", "p - kp + m", "mu"]], rules=rules)
T_su = form.compute(vectors=V, lines=[[pp, "nu", "p + k + m", "mu", "p + m", "nu", "p - kp + m", "mu"]], rules=rules)
M2 = (T_ss/(2*pk)^2 + T_uu/(2*pkp)^2 - 2*T_su/(4*pk*pkp)).simplify_full()    # sum |M|^2 / e^4
peskin = 8*(pkp/pk + pk/pkp + 2*m^2*(1/pk - 1/pkp) + m^4*(1/pk - 1/pkp)^2)   # 4 x Peskin (5.87)
print("sum |M|^2/e^4 minus 4 x Peskin (5.87):", (M2 - peskin).simplify_full())

sum |M|^2/e^4 minus 4 x Peskin (5.87): 0


**Klein–Nishina.** In the lab frame (electron at rest, photon energy $\omega$, angle $\theta$, $c = \cos\theta$)
$p\cdot k = m\omega$, $p\cdot k' = m\omega'$ with $\omega' = \omega/(1 + \frac{\omega}{m}(1 - c))$ and
$d\sigma/d\Omega = \frac{1}{64\pi^2 m^2}(\omega'/\omega)^2\cdot\frac14\sum|\mathcal M|^2$. For $\omega \to 0$ it must give the
Thomson cross section $8\pi\alpha^2/3m^2 = 8\pi r_0^2/3$:

In [3]:
w, c, alpha = var('w c alpha')
wp = w/(1 + w/m*(1 - c))
dsigma = ((wp/w)^2 * M2/4 * (4*pi*alpha)^2 / (64*pi^2*m^2)).subs(pk=m*w, pkp=m*wp)
klein_nishina = alpha^2/(2*m^2)*(wp/w)^2*(wp/w + w/wp - (1 - c^2))
print("dsigma/dOmega minus Klein-Nishina:", (dsigma - klein_nishina).simplify_full())
print("Thomson limit: sigma =", integrate(klein_nishina.subs(w=0)*2*pi, c, -1, 1))

dsigma/dOmega minus Klein-Nishina: 0
Thomson limit: sigma = 8/3*pi*alpha^2/m^2


## 2. Double Compton scattering: six diagrams

$e(P) + \gamma(K_0) \to e(P') + \gamma(K_1) + \gamma(K_2)$. The three photons attach to the electron line in
all $3! = 6$ orders. The thesis quotes Mandl and Skyrme's result as $|\mathcal M|^2 = e^6 X$ (its eq. D.1).

Six diagrams give $6\times 6 = 36$ traces, each with its own denominators. This bookkeeping is in
`double_compton.py` in this folder (read it for the details). Its functions:
- `dc_table(pk0, pk1, pk2, k0k1, k0k2)`: all scalar products from $P\cdot K_0$, $P\cdot K_1$, $P\cdot K_2$,
  $K_0\cdot K_1$, $K_0\cdot K_2$, with $m = 1$ (the on-shell $P'$ fixes the rest)
- `dc_msq(table)`: the sum over spins and polarisations of $|\mathcal M|^2/e^6$
- `mandl_skyrme_X(table)`: Mandl and Skyrme's $X$
- `eikonal(table)`: the soft-photon factor of $K_2$ (Section 3)

At four kinematic points with exact fractions:

In [4]:
for point in [(7/3, 26/25, 4/7, 35/8, 24/23), (4/21, 7/3, 1/3, 27/7, 16/7),
              (5/6, 2/9, 3/11, 7/13, 1/5), (9/4, 11/8, 13/12, 17/10, 3/7)]:
    tab = dc_table(*point)
    print(point, "  sum|M|^2 / (e^6 X) =", dc_msq(tab) / mandl_skyrme_X(tab))

(7/3, 26/25, 4/7, 35/8, 24/23)   sum|M|^2 / (e^6 X) = 4


(4/21, 7/3, 1/3, 27/7, 16/7)   sum|M|^2 / (e^6 X) = 4


(5/6, 2/9, 3/11, 7/13, 1/5)   sum|M|^2 / (e^6 X) = 4


(9/4, 11/8, 13/12, 17/10, 3/7)   sum|M|^2 / (e^6 X) = 4


Exactly 4 every time: the thesis's $|\mathcal M|^2 = e^6 X$ is the average over the $2\times 2$ spins and
polarisations of the initial electron and photon.

## 3. The soft limit: the infrared divergence

When $K_2 \to 0$ double Compton becomes Compton scattering times the *eikonal factor*
$$S = \frac{2P\cdot P'}{(P\cdot K_2)(P'\cdot K_2)} - \frac{m^2}{(P\cdot K_2)^2} - \frac{m^2}{(P'\cdot K_2)^2} \propto \frac{1}{\omega_2^2}$$
so $d\sigma \propto d\omega_2/\omega_2$: the divergence of the thesis (its section 4.4.5). Below, $K_2$ is scaled by $\lambda$
and the ratio must go to 1:

In [5]:
compton = M2.subs(m=1, pk=3/4, pkp=1/3)                 # Compton scattering at P.K0 = 3/4, P.K1 = 1/3
for lam in [1/10^4, 1/10^6, 1/10^8]:
    tab = dc_table(3/4, 1/3, lam/2, 3/4 - 1/3 - lam/5, lam/3)    # the photon K2 scaled by lam
    print("lambda = %g:   sum|M_DC|^2 / (S sum|M_C|^2) = %.10f" % (lam, dc_msq(tab) / (eikonal(tab)*compton)))

lambda = 0.0001:   sum|M_DC|^2 / (S sum|M_C|^2) = 0.9969856372


lambda = 1e-06:   sum|M_DC|^2 / (S sum|M_C|^2) = 0.9999698113


lambda = 1e-08:   sum|M_DC|^2 / (S sum|M_C|^2) = 0.9999996981


The divergence and how it cancels against the one-loop vertex are in `ir_cancellation.ipynb`.